# PCVRHyFormer 数据处理全量走读 Notebook

> **目标**：把 baseline `train/dataset.py`（763 行）逐段中文讲解。
> 本 notebook 是 [`model_full_walkthrough.ipynb`](model_full_walkthrough.ipynb) 的姊妹篇 —— 那个 notebook 把模型 16 个类原样搬入；
> 本 notebook 把数据流的每一个环节拆开讲清楚。
>
> **覆盖范围**：
> 1. `FeatureSchema` —— `(fid, offset, length)` 切片管理
> 2. `BUCKET_BOUNDARIES` —— 时间桶设计哲学
> 3. `PCVRParquetDataset.__init__` —— 4 步初始化（Row Group 收集 / schema 加载 / 缓冲区预分配 / 列计划预编排）
> 4. `_load_schema` —— 5 大特征组（user_int / item_int / user_dense / item_dense / seq）从 schema.json 还原
> 5. `_pad_varlen_int_column` / `_pad_varlen_float_column` —— 变长列对齐
> 6. `_record_oob` / `dump_oob_stats` —— OOB（越界 vocab id）容错
> 7. `_convert_batch` —— **核心**！Arrow RecordBatch → 训练可用 dict（含 meta、user_int、item_int、user_dense、序列特征三层 padding、时间桶生成）
> 8. `__iter__` + `_flush_buffer` —— Row Group 流式读取 + window-shuffle 双层缓冲
> 9. `get_pcvr_data` —— Train/Valid 双 DataLoader 工厂（按 Row Group 切分）
> 10. **端到端实操** —— 合成一份微型 parquet + schema → 跑通完整流水线 → 打印每一项 shape

**前置阅读**：[`README_data.md`](sample_data/README_data.md) 中关于原始数据格式的描述（120 列、5 大特征组、序列域定义）。

## Step 0 · 环境与依赖

把 `train/` 加入 `sys.path`，方便我们直接 import baseline 的 `dataset` 模块对照运行。
我们既会**讲解**源码，又会**真的执行**关键函数，所以两者都要导入。

In [1]:
import sys, os, json, logging, random, glob, gc
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch.utils.data import IterableDataset, DataLoader

WORK_DIR = Path(os.getcwd())
TRAIN_DIR = WORK_DIR / 'train'
SAMPLE_DIR = WORK_DIR / 'sample_data'
sys.path.insert(0, str(TRAIN_DIR))

# 关键：导入 baseline 原版 dataset 模块对照
import dataset as bl_dataset  # noqa: E402
print('baseline dataset module:', bl_dataset.__file__)
print('NUM_TIME_BUCKETS =', bl_dataset.NUM_TIME_BUCKETS, '(= len(BUCKET_BOUNDARIES) + 1)')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

baseline dataset module: /Users/huangjing103/PyCharmMiscProject/taac-codes/train/dataset.py
NUM_TIME_BUCKETS = 64 (= len(BUCKET_BOUNDARIES) + 1)


## Section 1 · `FeatureSchema` —— `(fid, offset, length)` 切片管理类

### 1.1 它要解决什么问题？

模型侧把所有 `user_int_feats` **拼成一个 flat tensor**（形状 `(B, total_dim)`），但实际有几十个 fid，每个 fid 占其中**一段**。
谁占 `[0:1]`、谁占 `[1:5]`、谁占 `[5:6]` …… 必须有个表记下来。这就是 `FeatureSchema`。

### 1.2 数据结构 3 件套
- `entries: List[(fid, offset, length)]` —— 顺序记录每个 fid 的切片
- `total_dim` —— 当前已分配的总宽度（用作下一个 fid 的 offset）
- `_fid_to_entry: Dict[fid, (offset, length)]` —— O(1) 查找用

### 1.3 length 的语义随特征类型而变
- `int_value`（标量）→ `length = 1`
- `int_array`（multi-hot）→ `length = 数组上限`
- `float_value` / `float_array` → 同上
- `int_array_and_float_array` → 拆 int 部分长度 / float 部分长度

### 1.4 序列化（`to_dict` / `from_dict`）

存盘时只存 `entries` 和 `total_dim` 两个字段；`_fid_to_entry` 在 `from_dict` 时重建。
这样 `schema.json` 体积小且可读性好。

In [2]:
# === baseline train/dataset.py 第 43-103 行 === （原样搬运 + 注释）
class FeatureSchema:
    """Records ``(feature_id, offset, length)`` for each feature so downstream
    code can locate the segment of the flattened tensor that belongs to a
    specific feature id."""

    def __init__(self) -> None:
        self.entries: List[Tuple[int, int, int]] = []
        self.total_dim: int = 0
        self._fid_to_entry: Dict[int, Tuple[int, int]] = {}

    def add(self, feature_id: int, length: int) -> None:
        """Append a feature to the schema.

        关键点：offset = 当前 total_dim（上一个特征结束的位置）；
        然后 total_dim += length（为下一个特征让位）。
        所以 add 顺序就是 flat tensor 的拼接顺序。"""
        offset = self.total_dim
        self.entries.append((feature_id, offset, length))
        self._fid_to_entry[feature_id] = (offset, length)
        self.total_dim += length

    def get_offset_length(self, feature_id: int) -> Tuple[int, int]:
        return self._fid_to_entry[feature_id]

    @property
    def feature_ids(self) -> List[int]:
        return [fid for fid, _, _ in self.entries]

    def to_dict(self) -> Dict[str, Any]:
        return {'entries': self.entries, 'total_dim': self.total_dim}

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'FeatureSchema':
        schema = cls()
        for fid, offset, length in d['entries']:
            schema.entries.append((fid, offset, length))
            schema._fid_to_entry[fid] = (offset, length)
        schema.total_dim = d['total_dim']
        return schema

    def __repr__(self) -> str:
        lines = [f'FeatureSchema(total_dim={self.total_dim}, features=[']
        for fid, offset, length in self.entries:
            lines.append(f'  fid={fid}: offset={offset}, length={length}')
        lines.append('])')
        return '\n'.join(lines)


# ===== 实操演示 =====
schema = FeatureSchema()
schema.add(feature_id=101, length=1)    # 标量 int 特征
schema.add(feature_id=102, length=5)    # multi-hot, 上限 5
schema.add(feature_id=103, length=1)    # 标量
print(schema)
print('查询 fid=102 的切片:', schema.get_offset_length(102))
print('total_dim:', schema.total_dim, '→ flat tensor 形状 (B, 7)')

FeatureSchema(total_dim=7, features=[
  fid=101: offset=0, length=1
  fid=102: offset=1, length=5
  fid=103: offset=6, length=1
])
查询 fid=102 的切片: (1, 5)
total_dim: 7 → flat tensor 形状 (B, 7)


## Section 2 · `BUCKET_BOUNDARIES` + `NUM_TIME_BUCKETS` —— 时间桶设计

### 2.1 为什么要分桶？

每个时间步上的 photo 距离当次曝光时间多久？这个 `time_diff` 是**连续秒数**（0 ~ 几亿秒），
直接喂模型让模型很难学。所以做法是：把秒数离散成 64 个**人类感知尺度**的桶（"几秒前"、"几小时前"、"几天前"），
为每个桶学一个 d_model 维向量加到该位置的序列表征上 —— 这是经典的"几何级数桶分箱"trick。

### 2.2 边界设计哲学：前密后疏
- 1~60 秒：12 个桶（5 秒一档）
- 2~10 分钟：9 个桶
- 15~60 分钟：10 个桶
- 1.5~6 小时：10 个桶
- 9~24 小时：6 个桶
- 2~7 天：6 个桶
- 13~30 天 / 50~90 天 / 135~180 天 / 1 年：4+3+2+1 个

直觉：推荐场景里"刚刚发生 vs 一小时前"差异远大于"3 个月前 vs 3 个月+1 天前"。

### 2.3 桶号约定
- **0 桶留给 padding**（时间戳为 0 的位置）
- **1~64 桶**是真实桶
- 总数 65 = `len(BUCKET_BOUNDARIES) + 1`
- 模型侧 `nn.Embedding(NUM_TIME_BUCKETS=65, d_model, padding_idx=0)` 严格匹配

In [3]:
# === baseline train/dataset.py 第 109-132 行 ===
BUCKET_BOUNDARIES = np.array([
    5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60,           # 1~60 秒
    120, 180, 240, 300, 360, 420, 480, 540, 600,             # 2~10 分钟
    900, 1200, 1500, 1800, 2100, 2400, 2700, 3000, 3300, 3600,  # 15~60 分钟
    5400, 7200, 9000, 10800, 12600, 14400, 16200, 18000, 19800, 21600,  # 1.5~6 小时
    32400, 43200, 54000, 64800, 75600, 86400,                # 9~24 小时
    172800, 259200, 345600, 432000, 518400, 604800,          # 2~7 天
    1123200, 1641600, 2160000, 2592000,                      # 13~30 天
    4320000, 6048000, 7776000,                               # 50~90 天
    11664000, 15552000,                                      # 135~180 天
    31536000,                                                # 1 年
], dtype=np.int64)
NUM_TIME_BUCKETS = len(BUCKET_BOUNDARIES) + 1   # = 65 (含桶 0)

print(f'边界数量: {len(BUCKET_BOUNDARIES)}, 桶总数: {NUM_TIME_BUCKETS}')
print(f'最小边界: {BUCKET_BOUNDARIES[0]} 秒, 最大边界: {BUCKET_BOUNDARIES[-1]} 秒 (= {BUCKET_BOUNDARIES[-1]/86400:.0f} 天)')

# === 实操：演示几个典型 time_diff 落到哪个桶 ===
demos = [3, 7, 60, 600, 3600, 86400, 86400*7, 86400*30, 86400*365, 86400*365*2]
labels = ['3 秒', '7 秒', '60 秒', '10 分钟', '1 小时', '1 天', '1 周', '30 天', '1 年', '2 年（极端）']
print('\n=== 时间差 → 桶号映射演示 ===')
for td, lb in zip(demos, labels):
    raw = np.searchsorted(BUCKET_BOUNDARIES, td)        # [0, 64]
    raw = min(raw, len(BUCKET_BOUNDARIES) - 1)          # clip 到 [0, 63]
    bucket = raw + 1                                      # [1, 64]
    print(f'  time_diff = {td:>10d} 秒 ({lb:>10s}) → bucket = {bucket}')

边界数量: 63, 桶总数: 64
最小边界: 5 秒, 最大边界: 31536000 秒 (= 365 天)

=== 时间差 → 桶号映射演示 ===
  time_diff =          3 秒 (       3 秒) → bucket = 1
  time_diff =          7 秒 (       7 秒) → bucket = 2
  time_diff =         60 秒 (      60 秒) → bucket = 12
  time_diff =        600 秒 (     10 分钟) → bucket = 21
  time_diff =       3600 秒 (      1 小时) → bucket = 31
  time_diff =      86400 秒 (       1 天) → bucket = 47
  time_diff =     604800 秒 (       1 周) → bucket = 53
  time_diff =    2592000 秒 (      30 天) → bucket = 57
  time_diff =   31536000 秒 (       1 年) → bucket = 63
  time_diff =   63072000 秒 (   2 年（极端）) → bucket = 63


## Section 3 · `PCVRParquetDataset.__init__` —— 4 步初始化

继承自 `IterableDataset` 而非 `Dataset` —— 因为 parquet 是**流式按 Row Group 读取**，
不能像 indexed dataset 那样随机 `__getitem__`。

### 3.1 `__init__` 4 个核心步骤

| 步骤 | 做什么 | 原因 |
|---|---|---|
| ① 收集 Row Groups | 遍历所有 `*.parquet`，记录 `(file, rg_idx, num_rows)` | DataLoader 多 worker 时用 RG 切分 |
| ② 加载 schema.json | 5 大特征组的 `(fid, vocab_size, dim)` 列表 | 决定每个 buffer 的形状 |
| ③ 预分配 numpy 缓冲区 | `_buf_user_int`、`_buf_seq` 等 | **零拷贝优化**：避免每个 batch 都 `np.zeros` |
| ④ 预编排列计划 | `_user_int_plan = [(col_idx, dim, offset, vocab_size), ...]` | **避免每行字符串查列名**：直接用 col_idx |

### 3.2 关键参数
- `batch_size`：固定值，所有缓冲区按这个 B 预分配
- `seq_max_lens`：可按 domain 覆盖序列截断长度（默认 256）
- `shuffle` + `buffer_batches`：window-shuffle 的窗口大小（见 Section 8）
- `row_group_range`：`(start, end)` 切片，用于 train/valid 划分
- `clip_vocab`：True 时把 OOB id 强制改成 0；False 时直接 raise
- `is_training`：True 时 `label = (label_type == 2).astype(int64)`；False 时全 0

### 3.3 性能优化亮点
- `torch.multiprocessing.set_sharing_strategy('file_system')`：用文件系统而非 `/dev/shm` 共享 tensor，避免共享内存被打满（多 worker 场景）
- `_buf_seq[domain]: shape (B, n_feats, max_len)`：3D 缓冲，**避免 padding loop 后再 stack**
- `_col_idx`：把 schema 列名映射成索引，每行直接 `batch.column(ci)` 而不是 `batch.column(name)`

In [4]:
# === baseline train/dataset.py 第 145-270 行（PCVRParquetDataset.__init__ 节选）===
# 这里**不**重新定义类，而是去 dive 到 baseline 已经实例化的 PCVRParquetDataset
# 看每个属性长什么样。先跳过实例化（需要 schema.json），等到 Section 10 再跑。

# 关键代码逻辑速览：
import inspect
src = inspect.getsource(bl_dataset.PCVRParquetDataset.__init__)
print('=== PCVRParquetDataset.__init__ 源码（前 30 行预览）===\n')
for i, line in enumerate(src.split('\n')[:30], 1):
    print(f'{i:3d} | {line}')
print('... (完整源码见 train/dataset.py 第 145-270 行)')

=== PCVRParquetDataset.__init__ 源码（前 30 行预览）===

  1 |     def __init__(
  2 |         self,
  3 |         parquet_path: str,
  4 |         schema_path: str,
  5 |         batch_size: int = 256,
  6 |         seq_max_lens: Optional[Dict[str, int]] = None,
  7 |         shuffle: bool = True,
  8 |         buffer_batches: int = 20,
  9 |         row_group_range: Optional[Tuple[int, int]] = None,
 10 |         clip_vocab: bool = True,
 11 |         is_training: bool = True,
 12 |     ) -> None:
 13 |         """
 14 |         Args:
 15 |             parquet_path: either a directory containing ``*.parquet`` files or
 16 |                 a single parquet file path.
 17 |             schema_path: path of the schema JSON describing feature layouts.
 18 |             batch_size: fixed batch size used for the pre-allocated buffers.
 19 |             seq_max_lens: optional per-domain override of sequence truncation,
 20 |                 e.g. ``{'seq_d': 256}``. Domains not listed fall back to 

## Section 4 · `_load_schema` —— schema.json 的 5 大特征组解析

### 4.1 schema.json 结构

```json
{
  "user_int":   [[fid, vocab_size, dim], ...],   // 用户侧整型特征
  "item_int":   [[fid, vocab_size, dim], ...],   // 物料侧整型特征
  "user_dense": [[fid, dim], ...],                // 用户侧浮点特征（dense feature 没有 vocab）
  "seq": {
    "seq_a": {                                    // 序列 domain 'seq_a'
      "prefix": "user_int_seq_a_feats",          // parquet 列名前缀
      "ts_fid": 1,                                // 哪个 fid 是时间戳列
      "features": [[fid, vocab_size], ...]        // 该 domain 下所有 fid（含 ts_fid）
    },
    ...
  }
}
```

### 4.2 解析后落到 dataset 实例的属性
| 属性 | 类型 | 说明 |
|---|---|---|
| `_user_int_cols` | `List[(fid, vs, dim)]` | 顺序原始列表 |
| `user_int_schema` | `FeatureSchema` | flat tensor 切片表 |
| `user_int_vocab_sizes` | `List[int]`（长度=total_dim） | 每个 slot 对应的 vocab 大小，**每个 fid 重复 dim 次** |
| `seq_domains` | `List[str]`（已 sorted）| 序列域名字 |
| `seq_vocab_sizes` | `Dict[domain, Dict[fid, vs]]` | 二级查询表 |
| `sideinfo_fids` | `Dict[domain, List[fid]]` | 排除 ts_fid 后的 fid 顺序 |
| `_seq_maxlen` | `Dict[domain, int]` | 每个 domain 的 padding 上限（默认 256）|

### 4.3 注意点
- **`vs == 0`** 表示该特征没有 vocab 信息 → dataset 端会强制把所有值清 0，模型端只给它 1 个 Embedding slot（避免越界）
- **`item_dense` 是空的** —— baseline 数据集里没有 item 侧的 dense 特征，但代码留了占位（`item_dense_schema = FeatureSchema()`）
- **dense 特征的 vocab_size 不需要**（dense 是浮点数）

In [5]:
# === baseline train/dataset.py 第 272-330 行 _load_schema ===
src = inspect.getsource(bl_dataset.PCVRParquetDataset._load_schema)
print('=== _load_schema 源码 ===\n')
print(src)

=== _load_schema 源码 ===

    def _load_schema(self, schema_path: str, seq_max_lens: Dict[str, int]) -> None:
        """Populate per-group schema information from ``schema_path``."""
        with open(schema_path, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        # ---- user_int: [[fid, vocab_size, dim], ...] ----
        self._user_int_cols: List[List[int]] = raw['user_int']
        self.user_int_schema: FeatureSchema = FeatureSchema()
        self.user_int_vocab_sizes: List[int] = []
        for fid, vs, dim in self._user_int_cols:
            self.user_int_schema.add(fid, dim)
            self.user_int_vocab_sizes.extend([vs] * dim)

        # ---- item_int ----
        self._item_int_cols: List[List[int]] = raw['item_int']
        self.item_int_schema: FeatureSchema = FeatureSchema()
        self.item_int_vocab_sizes: List[int] = []
        for fid, vs, dim in self._item_int_cols:
            self.item_int_schema.add(fid, dim)
            self.item_int_vocab_size

## Section 5 · 变长列 padding 工具

`_pad_varlen_int_column` / `_pad_varlen_float_column` 是把 Arrow `ListArray<int>` / `ListArray<float>` 对齐成 `(B, max_len)` numpy 矩阵的核心工具。

### 5.1 Arrow ListArray 长什么样？

每条记录的"列表特征"在 Arrow 里以 **(offsets, values) 二元组**存储（避免变长嵌套的内存碎片）：

```
原始数据：
  样本 0: [10, 20, 30]
  样本 1: [40]
  样本 2: [50, 60]

Arrow 表示：
  offsets = [0, 3, 4, 6]               # 每条样本在 values 里的起止位置
  values  = [10, 20, 30, 40, 50, 60]   # 所有样本的值平铺
```

第 i 条样本的值 = `values[offsets[i]:offsets[i+1]]`

### 5.2 padding 的 3 个细节
- **values <= 0 都被映射成 0** —— 原始数据里 `-1` 表示 missing，这里和 0（=padding）做相同处理
- **超过 `max_len` 的部分被截断** —— `use_len = min(raw_len, max_len)`
- **`lengths[i]`** 同时记下真实有效长度，给后面的 attention mask 用

### 5.3 int / float 版本的差异
- int 版返回 `(padded, lengths)` 两个东西（外面要用 lengths 做 attention mask）
- float 版只返回 `padded`（dense 特征不需要 mask，0 填充就是合理的 default）

In [6]:
# === baseline train/dataset.py 第 445-503 行 ===
src1 = inspect.getsource(bl_dataset.PCVRParquetDataset._pad_varlen_int_column)
src2 = inspect.getsource(bl_dataset.PCVRParquetDataset._pad_varlen_float_column)
print('=== _pad_varlen_int_column ===')
print(src1)
print('\n=== _pad_varlen_float_column ===')
print(src2)

# === 实操：用 pyarrow 构造一个 ListArray 跑一遍 ===
print('\n=== 实操：变长 padding 演示 ===')
arr = pa.array([[10, 20, 30], [40], [50, 60]], type=pa.list_(pa.int64()))
print('Arrow ListArray:')
print('  offsets:', arr.offsets.to_numpy().tolist())
print('  values: ', arr.values.to_numpy().tolist())

# 直接调用 baseline 的方法
# 但 _pad_varlen_int_column 是实例方法，需要先有实例 —— 这里手工模拟其逻辑
B = 3
max_len = 5
offsets = arr.offsets.to_numpy()
values = arr.values.to_numpy()
padded = np.zeros((B, max_len), dtype=np.int64)
lengths = np.zeros(B, dtype=np.int64)
for i in range(B):
    s, e = int(offsets[i]), int(offsets[i + 1])
    raw_len = e - s
    if raw_len <= 0:
        continue
    use_len = min(raw_len, max_len)
    padded[i, :use_len] = values[s:s + use_len]
    lengths[i] = use_len
padded[padded <= 0] = 0
print(f'\npadded (shape={padded.shape}):')
print(padded)
print(f'lengths: {lengths.tolist()}')

=== _pad_varlen_int_column ===
    def _pad_varlen_int_column(
        self,
        arrow_col: "pa.ListArray",
        max_len: int,
        B: int,
    ) -> Tuple["npt.NDArray[np.int64]", "npt.NDArray[np.int64]"]:
        """Pad an Arrow ``ListArray`` of ints to shape ``[B, max_len]``.

        Values <= 0 are mapped to 0 (padding). Note: the raw data contains -1
        (missing); currently treated the same way as 0 (padding).

        Returns:
            A tuple ``(padded, lengths)`` where ``padded`` has shape
            ``[B, max_len]`` and ``lengths`` has shape ``[B]``.
        """
        offsets = arrow_col.offsets.to_numpy()
        values = arrow_col.values.to_numpy()

        padded = np.zeros((B, max_len), dtype=np.int64)
        lengths = np.zeros(B, dtype=np.int64)

        for i in range(B):
            start, end = int(offsets[i]), int(offsets[i + 1])
            raw_len = end - start
            if raw_len <= 0:
                continue
            use_len = min(raw_

Arrow ListArray:
  offsets: [0, 3, 4, 6]
  values:  [10, 20, 30, 40, 50, 60]

padded (shape=(3, 5)):
[[10 20 30  0  0]
 [40  0  0  0  0]
 [50 60  0  0  0]]
lengths: [3, 1, 2]


## Section 6 · `_record_oob` + `dump_oob_stats` —— OOB 容错与统计

### 6.1 OOB 是什么？

模型侧每个 fid 学一个 `nn.Embedding(vocab_size, emb_dim)`，**id 必须在 `[0, vocab_size)` 范围内**，否则会抛 `IndexError: index out of range in self`。
但实际数据脏，可能有：
- 数据生成 bug，吐出超过 vocab_size 的 id
- 增量训练时 vocab 扩了但 schema.json 没同步
- 测试集里出现训练集没见过的新 id

### 6.2 baseline 的 OOB 容错策略
- `clip_vocab=True`（默认）：把 OOB 值**强制改为 0**（即 padding），训练继续
- `clip_vocab=False`：直接 `raise ValueError`，看哪个 fid / col 越界了

### 6.3 `_oob_stats` 累计统计
```
_oob_stats: {(group, col_idx): {'count': N, 'max': M, 'min_oob': M, 'vocab': V}}
```
最后调用 `dump_oob_stats(path)` 把所有 OOB 信息写到文件，方便排查。

In [7]:
# === baseline train/dataset.py 第 388-443 行 ===
src1 = inspect.getsource(bl_dataset.PCVRParquetDataset._record_oob)
src2 = inspect.getsource(bl_dataset.PCVRParquetDataset.dump_oob_stats)
print('=== _record_oob ===')
print(src1)
print('\n=== dump_oob_stats ===')
print(src2)

=== _record_oob ===
    def _record_oob(
        self,
        group: str,
        col_idx: int,
        arr: "npt.NDArray[np.int64]",
        vocab_size: int,
    ) -> None:
        """Record out-of-bound indices and (optionally) clip them to 0,
        without printing to the console.
        """
        oob_mask = arr >= vocab_size
        if not oob_mask.any():
            return
        key = (group, col_idx)
        oob_vals = arr[oob_mask]
        n = int(oob_mask.sum())
        mx = int(oob_vals.max())
        mn = int(oob_vals.min())
        if key in self._oob_stats:
            s = self._oob_stats[key]
            s['count'] += n
            s['max'] = max(s['max'], mx)
            s['min_oob'] = min(s['min_oob'], mn)
        else:
            self._oob_stats[key] = {
                'count': n, 'max': mx, 'min_oob': mn, 'vocab': vocab_size,
            }
        if self.clip_vocab:
            arr[oob_mask] = 0
        else:
            raise ValueError(
                f"{

## Section 7 · `_convert_batch` —— **核心**：Arrow RecordBatch → 训练可用 dict

这是整个 dataset.py 最重要的函数（160+ 行）。它把一个 Arrow `RecordBatch`（约 256 行）
转换成一个 dict，里面 key 对应模型侧 `ModelInput` 字段。我们按 5 个子段拆开讲。

### 7.1 子段 A：meta + label

```python
timestamps = batch.column('timestamp').to_numpy().astype(int64)
labels = (batch.column('label_type').fill_null(0)
          .to_numpy(zero_copy_only=False).astype(int64) == 2).astype(int64)
user_ids = batch.column('user_id').to_pylist()
```

**关键点**：
- `label_type` 在原始数据里是多分类（0=曝光, 1=点击, 2=转化, ...）
- baseline 的任务是 **CVR（conversion rate）**，所以 `label = (label_type == 2)` —— 这是 PCVR 项目名的由来
- `is_training=False`（推理）时直接给 全 0 label
- `user_ids` 用 Python list 而不是 tensor —— 字符串型 ID

### 7.2 子段 B：user_int / item_int 整型特征

```python
for ci, dim, offset, vs in self._user_int_plan:
    col = batch.column(ci)
    if dim == 1:
        # 标量整型：直接拿值
        arr = col.fill_null(0).to_numpy(zero_copy_only=False).astype(int64)
        arr[arr <= 0] = 0   # null & -1 → 0
        if vs > 0:
            self._record_oob('user_int', ci, arr, vs)   # OOB 检测
        else:
            arr[:] = 0      # 没 vocab 信息，强制清 0
        user_int[:, offset] = arr
    else:
        # multi-hot 数组：调 _pad_varlen_int_column
        padded, _ = self._pad_varlen_int_column(col, dim, B)
        ...
        user_int[:, offset:offset+dim] = padded
```

`_user_int_plan` 是 `__init__` 时预编排的 `[(col_idx, dim, offset, vocab_size), ...]`，
避免每次都用列名字符串查 col_idx。

### 7.3 子段 C：user_dense 浮点特征

```python
for ci, dim, offset in self._user_dense_plan:
    col = batch.column(ci)
    padded = self._pad_varlen_float_column(col, dim, B)
    user_dense[:, offset:offset+dim] = padded
```

dense 没有 vocab 概念，更简单 —— 只做变长 padding 就行。
（注：baseline 数据里 `item_dense` 是空的，所以代码里没有对应循环；模型侧 `item_dense_feats` 也是 `(B, 0)` 的空 tensor）

### 7.4 子段 D：序列特征 —— 三层 padding

最复杂的一段。每个 domain（如 'seq_a'）下有：
- 多个 **side-info fid**（如 photo_id / channel_id / cate_id）
- 1 个 **timestamp fid**

序列特征要打包成 3D 张量 `(B, n_feats, max_len)`，对齐方式：
- **第 0 维 B**：batch 内样本
- **第 1 维 n_feats**：side-info fid 顺序（time_stamp 不算）
- **第 2 维 max_len**：每个样本对齐到 batch 内统一长度

```python
out = self._buf_seq[domain][:B]         # 复用 3D 缓冲区
lengths = self._buf_seq_lens[domain][:B]

# Fused 路径：先收集所有 side-info 的 (offsets, values, vocab_size)
col_data = []
for ci, slot, vs in side_plan:
    col = batch.column(ci)
    col_data.append((col.offsets.to_numpy(), col.values.to_numpy(), vs, ci))

# 一次性写满 3D 缓冲区
for c, (offs, vals, vs, ci) in enumerate(col_data):
    for i in range(B):
        s, e = int(offs[i]), int(offs[i+1])
        ul = min(e - s, max_len)
        out[i, c, :ul] = vals[s:s+ul]
        if ul > lengths[i]:
            lengths[i] = ul              # 取所有 side-info 中**最长**的
```

**关键点**：
- 同一个 domain 下所有 side-info **共享 lengths**（取 max）—— 因为它们指向同一组时间步上的不同属性
- `out[out <= 0] = 0` —— null/-1 → padding

### 7.5 子段 E：时间桶生成

```python
ts_padded = np.zeros((B, max_len), dtype=np.int64)
# 把 timestamp 列变长 padding 到 (B, max_len)
for i in range(B):
    ...
    ts_padded[i, :ul] = ts_vals[s:s+ul]

ts_expanded = timestamps.reshape(-1, 1)              # (B, 1)：当次曝光时间
time_diff = np.maximum(ts_expanded - ts_padded, 0)   # (B, max_len)：每个时间步距当次曝光多少秒

raw = np.clip(np.searchsorted(BUCKET_BOUNDARIES, time_diff.ravel()),
              0, len(BUCKET_BOUNDARIES) - 1)         # 二分查找桶号
buckets = raw.reshape(B, max_len) + 1                # +1 给桶 0 留位（padding）
buckets[ts_padded == 0] = 0                          # 时间戳为 0 的位置回归桶 0
```

返回 `result['{domain}_time_bucket']`，模型侧 `nn.Embedding(NUM_TIME_BUCKETS, d_model, padding_idx=0)`
就用这个 id 查 embedding 加到序列向量上。

In [8]:
# === baseline train/dataset.py 第 505-669 行 _convert_batch ===
src = inspect.getsource(bl_dataset.PCVRParquetDataset._convert_batch)
print('=== _convert_batch 源码（共', len(src.split('\n')), '行）===\n')
print(src)

=== _convert_batch 源码（共 166 行）===

    def _convert_batch(self, batch: "pa.RecordBatch") -> Dict[str, Any]:
        """Convert an Arrow RecordBatch into a training-ready dict of tensors."""
        B = batch.num_rows

        # ---- meta ----
        timestamps = batch.column(self._col_idx['timestamp']).to_numpy().astype(np.int64)
        if self.is_training:
            labels = (batch.column(self._col_idx['label_type']).fill_null(0)
                      .to_numpy(zero_copy_only=False).astype(np.int64) == 2).astype(np.int64)
        else:
            labels = np.zeros(B, dtype=np.int64)
        user_ids = batch.column(self._col_idx['user_id']).to_pylist()

        # ---- user_int: write into pre-allocated buffer ----
        # Note: null -> 0 (via fill_null), -1 -> 0 (via arr<=0); missing values
        # are treated the same as padding. Features with vs==0 have no vocab
        # information and are forced to 0 on the dataset side so that the
        # model's 1-slot Embedding (crea

## Section 8 · `__iter__` + `_flush_buffer` —— Row Group 流式读取 + window-shuffle

### 8.1 多 worker 切分

DataLoader 启用 `num_workers=N` 时，每个 worker 都会调 `__iter__`。
为了避免数据重复，**按 worker_id 把 row group 列表分片**：

```python
worker_info = torch.utils.data.get_worker_info()
if worker_info is not None and worker_info.num_workers > 1:
    rg_list = [rg for i, rg in enumerate(rg_list)
               if i % worker_info.num_workers == worker_info.id]
```

### 8.2 流式读取主循环

```python
for file_path, rg_idx, _ in rg_list:
    pf = pq.ParquetFile(file_path)
    for batch in pf.iter_batches(batch_size=self.batch_size, row_groups=[rg_idx]):
        batch_dict = self._convert_batch(batch)
        if self.shuffle and self.buffer_batches > 1:
            buffer.append(batch_dict)
            if len(buffer) >= self.buffer_batches:
                yield from self._flush_buffer(buffer)
                buffer = []
        else:
            yield batch_dict
```

注意 **`pf.iter_batches(row_groups=[rg_idx])` 一次只读一个 RG**，内存可控；
不会一下把整个文件读进来。

### 8.3 window-shuffle 设计

baseline 的 shuffle 不是全量随机，而是 **滑动窗口随机**：
- 累积 `buffer_batches` 个 batch（例如 20 个）→ 一共 ~5120 行
- `_flush_buffer`：把这些 batch concat 起来，按行随机重排，再切回 batch 输出

**为什么不全量 shuffle？** —— 全量 shuffle 要把整个 dataset 加载进内存，对 10G+ parquet 不现实。
window-shuffle 在内存可控的前提下提供了**局部**的随机性，对训练效果通常已经够了。

### 8.4 valid 默认 `buffer_batches=0` + `shuffle=False`
直接顺序流式输出，最大化吞吐 + 确保 valid 结果可复现。

In [9]:
# === baseline train/dataset.py 第 337-385 行 ===
src1 = inspect.getsource(bl_dataset.PCVRParquetDataset.__iter__)
src2 = inspect.getsource(bl_dataset.PCVRParquetDataset._flush_buffer)
print('=== __iter__ ===')
print(src1)
print('\n=== _flush_buffer ===')
print(src2)

=== __iter__ ===
    def __iter__(self) -> Iterator[Dict[str, Any]]:
        worker_info = torch.utils.data.get_worker_info()
        rg_list = self._rg_list
        if worker_info is not None and worker_info.num_workers > 1:
            rg_list = [rg for i, rg in enumerate(rg_list)
                       if i % worker_info.num_workers == worker_info.id]

        buffer: List[Dict[str, Any]] = []
        for file_path, rg_idx, _ in rg_list:
            pf = pq.ParquetFile(file_path)
            for batch in pf.iter_batches(batch_size=self.batch_size, row_groups=[rg_idx]):
                batch_dict = self._convert_batch(batch)
                if self.shuffle and self.buffer_batches > 1:
                    buffer.append(batch_dict)
                    if len(buffer) >= self.buffer_batches:
                        yield from self._flush_buffer(buffer)
                        buffer = []
                else:
                    yield batch_dict

        if buffer:
            yield from

## Section 9 · `get_pcvr_data` —— Train/Valid 双 DataLoader 工厂

### 9.1 切分策略：按 Row Group 切

```python
n_valid_rgs = max(1, int(total_rgs * valid_ratio))
n_train_rgs = total_rgs - n_valid_rgs
```
- valid 取**末尾** `valid_ratio` 比例的 RG（典型 10%）
- 这是**时间序"前 N% 训练 / 后 (100-N)% 验证"**的常用做法

### 9.2 `train_ratio` 二次采样

如果只想用部分训练数据快速跑一下：
```python
if train_ratio < 1.0:
    n_train_rgs = max(1, int(n_train_rgs * train_ratio))
```

### 9.3 DataLoader 配置差异

| 配置项 | train | valid |
|---|---|---|
| shuffle | True | False |
| buffer_batches | 20（window-shuffle）| 0（顺序）|
| num_workers | 16（默认）| 0（避免 valid 卡顿）|
| persistent_workers | True | (无) |
| prefetch_factor | 2 | (无) |
| `batch_size=None` | ✓ | ✓ |

`batch_size=None` 是 IterableDataset 的标准用法 —— dataset 内部已经做好 batching，DataLoader 不再做 collate。

### 9.4 返回值
返回 `(train_loader, valid_loader, train_dataset)`，**第三个返回值很重要** —— 调用方需要从 `train_dataset` 拿到所有 schema 信息（`user_int_schema`、`seq_domains`、`sideinfo_fids` 等）来构造模型。

In [10]:
# === baseline train/dataset.py 第 672-763 行 ===
src = inspect.getsource(bl_dataset.get_pcvr_data)
print('=== get_pcvr_data 源码 ===\n')
print(src)

=== get_pcvr_data 源码 ===

def get_pcvr_data(
    data_dir: str,
    schema_path: str,
    batch_size: int = 256,
    valid_ratio: float = 0.1,
    train_ratio: float = 1.0,
    num_workers: int = 16,
    buffer_batches: int = 20,
    shuffle_train: bool = True,
    seed: int = 42,
    clip_vocab: bool = True,
    seq_max_lens: Optional[Dict[str, int]] = None,
    **kwargs: Any,
) -> Tuple[DataLoader, DataLoader, PCVRParquetDataset]:
    """Create train / valid DataLoaders from raw multi-column Parquet files.

    The validation split is taken as the last ``valid_ratio`` fraction of Row
    Groups (in the file order returned by ``glob``).

    Returns:
        A tuple ``(train_loader, valid_loader, train_dataset)``. The third
        element is returned so the caller can access the feature schema
        (``user_int_schema``, ``item_int_schema``, ...) needed to construct
        the model.
    """
    random.seed(seed)

    import glob as _glob
    pq_files = sorted(_glob.glob(os.path.j

## Section 10 · 端到端实操：合成微型 parquet → 跑通完整流水线

下面合成一个**微型** parquet（10 行，2 个序列 domain），然后调 baseline 的 `PCVRParquetDataset` 跑一个 batch 出来，
把每一项 shape 打印出来对照前面所有章节。

> 这里**不依赖** `sample_data/demo_1000.parquet`（用户本地 LFS 没拉到也能跑）；
> 但运行完之后你应该对 baseline 数据流的每一个细节都能心知肚明。

In [11]:
# === Step 1: 合成 schema（极简版本）===
WORK_TMP = WORK_DIR / 'playground_workdir' / 'dataset_walkthrough'
WORK_TMP.mkdir(parents=True, exist_ok=True)

schema_dict = {
    'user_int':   [[101, 1000, 1], [102, 500, 3]],   # fid=101 标量, fid=102 multi-hot 长度 3
    'item_int':   [[201, 2000, 1], [202, 100, 1]],   # 都是标量
    'user_dense': [[301, 4]],                         # fid=301 浮点数组长度 4
    'seq': {
        'seq_a': {
            'prefix': 'user_int_seq_a_feats',
            'ts_fid': 1,                              # fid=1 是 timestamp
            'features': [[1, 0], [10, 5000], [11, 200]],  # fid=10 是 photo_id, fid=11 是 cate_id
        },
    },
}
schema_path = WORK_TMP / 'schema.json'
with open(schema_path, 'w') as f:
    json.dump(schema_dict, f, indent=2)
print(f'已生成 schema → {schema_path}')

已生成 schema → /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir/dataset_walkthrough/schema.json


In [12]:
# === Step 2: 合成 10 行 parquet ===
N = 10
rng = np.random.RandomState(42)

# meta 列
data = {
    'user_id':    [f'u_{i:03d}' for i in range(N)],
    'timestamp':  np.array([1700000000 + i * 60 for i in range(N)], dtype=np.int64),
    'label_type': rng.choice([0, 1, 2], size=N).astype(np.int64),
}

# user_int_feats
data['user_int_feats_101'] = rng.randint(1, 1000, size=N).astype(np.int64)               # 标量
data['user_int_feats_102'] = [rng.randint(1, 500, size=rng.randint(1, 4)).tolist()       # 多值
                               for _ in range(N)]
# item_int
data['item_int_feats_201'] = rng.randint(1, 2000, size=N).astype(np.int64)
data['item_int_feats_202'] = rng.randint(1, 100, size=N).astype(np.int64)
# user_dense
data['user_dense_feats_301'] = [rng.uniform(0, 1, size=rng.randint(1, 5)).astype(np.float32).tolist()
                                 for _ in range(N)]
# 序列 seq_a：3 列（ts_fid=1, photo_id=10, cate_id=11），每行变长 5~30
for fid in [1, 10, 11]:
    col = []
    for i in range(N):
        L_i = rng.randint(5, 31)
        if fid == 1:
            # 时间戳：往前推（早于 timestamp）
            v = (data['timestamp'][i] - rng.randint(60, 86400, size=L_i)).astype(np.int64)
        elif fid == 10:
            v = rng.randint(1, 5000, size=L_i).astype(np.int64)
        else:
            v = rng.randint(1, 200, size=L_i).astype(np.int64)
        col.append(v.tolist())
    data[f'user_int_seq_a_feats_{fid}'] = col

table = pa.Table.from_pydict(data)
parquet_path = WORK_TMP / 'demo.parquet'
pq.write_table(table, parquet_path, row_group_size=N)  # 只 1 个 RG
print(f'已生成 parquet → {parquet_path}')
print(f'列数: {len(table.column_names)}, 列名: {table.column_names}')

已生成 parquet → /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir/dataset_walkthrough/demo.parquet
列数: 11, 列名: ['user_id', 'timestamp', 'label_type', 'user_int_feats_101', 'user_int_feats_102', 'item_int_feats_201', 'item_int_feats_202', 'user_dense_feats_301', 'user_int_seq_a_feats_1', 'user_int_seq_a_feats_10', 'user_int_seq_a_feats_11']


In [13]:
# === Step 3: 用 baseline 的 PCVRParquetDataset 跑一个 batch ===
ds = bl_dataset.PCVRParquetDataset(
    parquet_path=str(parquet_path),
    schema_path=str(schema_path),
    batch_size=4,                       # 故意比 N=10 小，看变长 padding 对齐到 batch 内最长序列
    seq_max_lens={'seq_a': 32},        # 截断上限 32
    shuffle=False,                      # 关 shuffle，确认顺序输出
    buffer_batches=0,
    is_training=True,
)
print('=== Dataset 信息 ===')
print(f'num_rows = {ds.num_rows}')
print(f'user_int_schema:    {ds.user_int_schema}')
print(f'item_int_schema:    {ds.item_int_schema}')
print(f'user_dense_schema:  {ds.user_dense_schema}')
print(f'seq_domains:        {ds.seq_domains}')
print(f'sideinfo_fids:      {ds.sideinfo_fids}')
print(f'ts_fids:            {ds.ts_fids}')
print(f'_seq_maxlen:        {ds._seq_maxlen}')

2026-04-30 10:28:09,967 - INFO - PCVRParquetDataset: 10 rows from 1 file(s), batch_size=4, buffer_batches=0, shuffle=False


=== Dataset 信息 ===
num_rows = 10
user_int_schema:    FeatureSchema(total_dim=4, features=[
  fid=101: offset=0, length=1
  fid=102: offset=1, length=3
])
item_int_schema:    FeatureSchema(total_dim=2, features=[
  fid=201: offset=0, length=1
  fid=202: offset=1, length=1
])
user_dense_schema:  FeatureSchema(total_dim=4, features=[
  fid=301: offset=0, length=4
])
seq_domains:        ['seq_a']
sideinfo_fids:      {'seq_a': [10, 11]}
ts_fids:            {'seq_a': 1}
_seq_maxlen:        {'seq_a': 32}


In [14]:
# === Step 4: 取出第一个 batch，逐项打印 shape ===
loader = DataLoader(ds, batch_size=None, num_workers=0)
batch = next(iter(loader))

print('=== Batch 内容（key → shape & dtype）===')
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:30s}: shape={str(tuple(v.shape)):20s} dtype={v.dtype}')
    elif isinstance(v, list):
        print(f'  {k:30s}: list, len={len(v)}, sample={v[:2]}')
    else:
        print(f'  {k:30s}: {type(v).__name__} = {v}')

print('\n=== label 分布 ===')
print(f'label = {batch["label"].tolist()}（baseline: label_type == 2 → 1）')

print('\n=== seq_a 第 0 条样本前 8 个时间步（photo_id 行）===')
print(f'photo_id: {batch["seq_a"][0, 0, :8].tolist()}')
print(f'cate_id : {batch["seq_a"][0, 1, :8].tolist()}')
print(f'time_bucket: {batch["seq_a_time_bucket"][0, :8].tolist()}')
print(f'seq_a_len: {batch["seq_a_len"][0].item()}')

=== Batch 内容（key → shape & dtype）===
  user_int_feats                : shape=(4, 4)               dtype=torch.int64
  user_dense_feats              : shape=(4, 4)               dtype=torch.float32
  item_int_feats                : shape=(4, 2)               dtype=torch.int64
  item_dense_feats              : shape=(4, 0)               dtype=torch.float32
  label                         : shape=(4,)                 dtype=torch.int64
  timestamp                     : shape=(4,)                 dtype=torch.int64
  user_id                       : list, len=4, sample=['u_000', 'u_001']
  _seq_domains                  : list, len=1, sample=['seq_a']
  seq_a                         : shape=(4, 2, 32)           dtype=torch.int64
  seq_a_len                     : shape=(4,)                 dtype=torch.int64
  seq_a_time_bucket             : shape=(4, 32)              dtype=torch.int64

=== label 分布 ===
label = [1, 0, 1, 1]（baseline: label_type == 2 → 1）

=== seq_a 第 0 条样本前 8 个时间步（photo_id 行）===

In [15]:
# === Step 5: 验证时间桶生成正确性 ===
# 取 batch[0] 的 timestamp 和 seq_a 的 ts_vals，手算 time_diff，对照 batch['seq_a_time_bucket']

i = 0
ts_now = int(batch['timestamp'][i])
seq_len = int(batch['seq_a_len'][i])

# 从 parquet 直接读这条样本的原始时间戳
orig = pq.read_table(parquet_path).to_pydict()
orig_ts = orig['user_int_seq_a_feats_1'][i][:seq_len]

print(f'第 {i} 条样本 ts_now = {ts_now}')
print(f'前 5 个历史时间戳: {orig_ts[:5]}')
print(f'对应 time_diff (秒): {[ts_now - t for t in orig_ts[:5]]}')

# 手算 bucket
expected = []
for diff in [ts_now - t for t in orig_ts[:5]]:
    raw = np.searchsorted(bl_dataset.BUCKET_BOUNDARIES, diff)
    raw = min(raw, len(bl_dataset.BUCKET_BOUNDARIES) - 1)
    expected.append(int(raw + 1))
print(f'手算 bucket: {expected}')
print(f'dataset 输出 bucket: {batch["seq_a_time_bucket"][i, :5].tolist()}')
print('\n✓ 一致！时间桶生成逻辑验证通过。')

第 0 条样本 ts_now = 1700000000
前 5 个历史时间戳: [1699997245, 1699951750, 1699994682, 1699960436, 1699966781]
对应 time_diff (秒): [2755, 48250, 5318, 39564, 33219]
手算 bucket: [29, 44, 32, 43, 43]
dataset 输出 bucket: [29, 44, 32, 43, 43]

✓ 一致！时间桶生成逻辑验证通过。


## 全文走读结束 ✓

到这里你已经：

1. **理解了 `FeatureSchema` 的切片管理设计** —— 解决"flat tensor 里谁占哪段"的问题
2. **理解了时间桶 64 个桶的几何级数设计** —— 前密后疏，padding=0 + 真实桶 1~64
3. **理解了 `PCVRParquetDataset.__init__` 的 4 步初始化** —— 性能优化全在 init 里（预分配缓冲、预编排列计划）
4. **理解了 `_load_schema` 5 大特征组的解析方式**
5. **理解了 Arrow ListArray 的 (offsets, values) 表示** + 变长 padding 工具
6. **理解了 OOB 容错策略** —— `clip_vocab=True` 默认行为
7. **完整走读了 `_convert_batch` 的 5 个子段** —— 从 Arrow 到训练 dict 的全过程
8. **理解了流式读取 + window-shuffle 的双层缓冲区设计**
9. **理解了 train/valid 按 Row Group 切分的工厂函数**
10. **真的合成了一份微型 parquet 并跑通了完整流水线**

### 与 model_full_walkthrough.ipynb 的关系

- 本 notebook 讲数据怎么从 parquet 流出来 → batch dict
- [`model_full_walkthrough.ipynb`](model_full_walkthrough.ipynb) 讲模型怎么吃 batch dict（先 wrap 成 `ModelInput` NamedTuple）→ 出 logits

两边对齐的接口：
| dataset 输出的 batch key | 对应 ModelInput 字段 |
|---|---|
| `user_int_feats` | `inputs.user_int_feats` |
| `item_int_feats` | `inputs.item_int_feats` |
| `user_dense_feats` | `inputs.user_dense_feats` |
| `item_dense_feats`（空 tensor）| `inputs.item_dense_feats` |
| `seq_a` | `inputs.seq_data['seq_a']`（trainer 内部组装）|
| `seq_a_len` | `inputs.seq_lens['seq_a']` |
| `seq_a_time_bucket` | `inputs.seq_time_buckets['seq_a']` |
| `label` | 训练时单独取出做 BCE loss |

trainer 在 `_make_model_input()`（[`train/trainer.py:376-398`](train/trainer.py:376-398)）做了这步转换。